In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv("../data/cleaned/playstore_features.csv")
df.columns = df.columns.str.replace(" ", "_")

conn = sqlite3.connect("../data/cleaned/playstore.db")
df.to_sql("apps", conn, if_exists="replace", index=False)

print(df.shape)

(9658, 22)


In [2]:
def q(sql):
    return pd.read_sql(sql, conn)

In [3]:
q("SELECT COUNT(*) AS total_apps FROM apps")

,total_apps
0,9658


In [4]:
q("""
SELECT COUNT(*)                                        AS total_apps,
       SUM(Success)                                    AS successful_apps,
       ROUND(100.0 * AVG(Success), 1)                  AS success_rate_pct,
       ROUND(AVG(Rating), 2)                           AS avg_rating,
       SUM(CASE WHEN Price = 0 THEN 1 ELSE 0 END)      AS free_apps
FROM apps
""")

,total_apps,successful_apps,success_rate_pct,avg_rating,free_apps
0,9658,3398,35.2,4.17,8904


In [5]:
q("""
SELECT Category,
       COUNT(*)                                        AS n_apps,
       SUM(Success)                                    AS n_success,
       ROUND(100.0 * AVG(Success), 1)                  AS success_rate_pct,
       ROUND(AVG(Rating), 2)                           AS avg_rating
FROM apps
GROUP BY Category
HAVING COUNT(*) >= 50
ORDER BY success_rate_pct DESC
""")

,Category,n_apps,n_success,success_rate_pct,avg_rating
0,ENTERTAINMENT,86,72,83.7,4.13
1,PHOTOGRAPHY,281,173,61.6,4.16
2,EDUCATION,105,61,58.1,4.35
3,GAME,945,539,57.0,4.24
4,WEATHER,79,42,53.2,4.24
5,SHOPPING,202,106,52.5,4.23
6,VIDEO_PLAYERS,164,85,51.8,4.04
7,COMMUNICATION,315,140,44.4,4.12
8,FOOD_AND_DRINK,112,49,43.8,4.17
9,TRAVEL_AND_LOCAL,219,93,42.5,4.07


In [6]:
q("""
WITH ranked AS (
    SELECT Category,
           Installs,
           ROW_NUMBER() OVER (PARTITION BY Category ORDER BY Installs DESC) AS rnk,
           SUM(Installs) OVER (PARTITION BY Category) AS category_total_installs
    FROM apps
)
SELECT Category,
       COUNT(*) AS n_apps,
       ROUND(100.0 * SUM(CASE WHEN rnk <= 10 THEN Installs ELSE 0 END)
             / MAX(category_total_installs), 1) AS top10_share_pct
FROM ranked
GROUP BY Category
HAVING COUNT(*) >= 50
ORDER BY top10_share_pct DESC
""")

,Category,n_apps,top10_share_pct
0,ART_AND_DESIGN,61,94.5
1,EVENTS,64,92.7
2,NEWS_AND_MAGAZINES,254,89.1
3,BOOKS_AND_REFERENCE,222,87.0
4,TRAVEL_AND_LOCAL,219,86.7
5,LIBRARIES_AND_DEMO,84,84.9
6,BEAUTY,53,84.6
7,SOCIAL,239,82.0
8,VIDEO_PLAYERS,164,80.1
9,COMICS,56,77.8


In [7]:
q("""
SELECT Price_Bucket,
       COUNT(*)                            AS n_apps,
       ROUND(100.0 * AVG(Success), 1)      AS success_rate_pct,
       ROUND(AVG(Rating), 2)               AS avg_rating
FROM apps
GROUP BY Price_Bucket
ORDER BY success_rate_pct DESC
""")

,Price_Bucket,n_apps,success_rate_pct,avg_rating
0,Free,8904,37.9,4.17
1,$5-10,84,4.8,4.26
2,$0.01-2,283,3.9,4.30
3,$2-5,314,2.2,4.26
4,$10+,73,0.0,4.13


In [8]:
q("""
SELECT Update_Bucket,
       COUNT(*)                            AS n_apps,
       ROUND(100.0 * AVG(Success), 1)      AS success_rate_pct
FROM apps
GROUP BY Update_Bucket
ORDER BY 
  CASE Update_Bucket
    WHEN '<=30d' THEN 1 WHEN '31-90d' THEN 2 WHEN '91-180d' THEN 3
    WHEN '181-365d' THEN 4 ELSE 5
  END
""")

,Update_Bucket,n_apps,success_rate_pct
0,<=30d,2936,56.7
1,31-90d,1828,38.9
2,91-180d,1158,23.5
3,181-365d,1320,22.9
4,>1yr,2416,18.5


In [9]:
q("""
SELECT Size_Bucket,
       COUNT(*)                            AS n_apps,
       ROUND(100.0 * AVG(Success), 1)      AS success_rate_pct
FROM apps
WHERE Size_Bucket IS NOT NULL
GROUP BY Size_Bucket
ORDER BY 
  CASE Size_Bucket
    WHEN '<5MB' THEN 1 WHEN '5-15MB' THEN 2 WHEN '15-30MB' THEN 3
    WHEN '30-60MB' THEN 4 ELSE 5
  END
""")

,Size_Bucket,n_apps,success_rate_pct
0,<5MB,2317,13.8
1,5-15MB,2494,25.5
2,15-30MB,1719,33.7
3,30-60MB,1301,46.5
4,60-100MB,600,58.3
